# Logging in Python (11/9/26 – 11/12/26)

---

*Codecademy — MLE Path: SWE for ML → **Logging**. Concise grad-student notes. Companion to [Functional Programming](Functional_Programming_Python_Lesson.ipynb).*

> **Scope:** why `logging` beats `print()`; creating a logger (`getLogger`, `StreamHandler`, `addHandler`); log levels and thresholds; level methods and `.log()`; `setLevel`; `FileHandler` and multiple handlers; `Formatter`/`setFormatter`; `basicConfig()`.

## Logging

### Introduction to Logging

Python's standard library ships a **`logging`** module for recording and tracking events in an application. Logged messages are the main tool for debugging and investigating behavior across development, testing, and production.

Why not just `print()`?

- **Clutter.** Prints pile up in the code, and in a complex app it's hard to tell where a given line of output came from.
- **Leftovers.** Debug prints tend to get forgotten and ship to production.
- **Formatting.** `logging` has built-in formatting options that make messages easy to read; formatting prints by hand is neither as simple nor as fast.

Clean, consistently formatted logs make the code easier to maintain, troubleshoot, and hand off to other developers.

**Example — a currency converter logged with `print()`.** Every message takes three hand-written prints: a `datetime.now()` timestamp, the module name (`__name__`, which is `"__main__"` when run directly), and a message with a hand-typed level prefix (`DEBUG:`, `INFO:`, `ERROR:`). Invalid currency codes print an error and `sys.exit(0)`.

That's the problem `logging` solves: the timestamp / source / level boilerplate is repeated at every call site and is easy to get inconsistent. A logger adds all three automatically from a single call. (Note `logging` is imported here but not used yet.)

In [1]:
import logging
import sys

from datetime import datetime

exchange_rates = {
  "USD": 1.0,
  "EUR": 0.861775,
  "GBP": 0.726763,
  "INR": 75.054725,
  "AUD": 1.333679,
  "CAD": 1.237816,
  "SGD": 1.346851
}

def convert(from_country, to_country, amount):
  if to_country not in exchange_rates:
    timestamp = datetime.now()
    print(timestamp)
    print(__name__)
    print("ERROR: The TO country supplied is not a valid country.")
    sys.exit(0)
    
  if from_country not in exchange_rates:
    timestamp = datetime.now()
    print(timestamp)
    print(__name__)
    print("ERROR: The FROM country supplied is not a valid country.")
    sys.exit(0)
        
  to_rate = exchange_rates[to_country]
  from_rate = exchange_rates[from_country]
    
  if from_country != "USD":
    converted_to_usd = amount / from_rate
    timestamp = datetime.now()
    print(timestamp)
    print(__name__)
    print("INFO: Converting from {from_country} to USD: {converted_to_usd}".format(from_country=from_country, converted_to_usd=converted_to_usd))
        
    converted_from_usd = converted_to_usd * to_rate
    timestamp = datetime.now()
    print(timestamp)
    print(__name__)
    print("INFO: Converting from USD to {to_country}: {converted_from_usd}".format(to_country=to_country, converted_from_usd=converted_from_usd))
        
    return converted_from_usd
        
  else:
    converted_from_usd = amount * to_rate
    timestamp = datetime.now()
    print (timestamp)
    print (__name__)
    print ("INFO: Converting from USD to {to_country}: {converted_from_usd}".format(to_country=to_country, converted_from_usd=converted_from_usd))
    return converted_from_usd
        
timestamp = datetime.now()
print(timestamp)
print(__name__)
print("DEBUG: Current rates: {exchange_rates}".format(exchange_rates=exchange_rates))        
currency = convert("EUR", "USD", 45)
print(currency)
# -> <timestamp> / __main__ / DEBUG: Current rates: {...}
# -> <timestamp> / __main__ / INFO: Converting from EUR to USD: 52.217806271938734
# -> <timestamp> / __main__ / INFO: Converting from USD to USD: 52.217806271938734
# -> 52.217806271938734

2026-11-12 17:49:59.797603
__main__
DEBUG: Current rates: {'USD': 1.0, 'EUR': 0.861775, 'GBP': 0.726763, 'INR': 75.054725, 'AUD': 1.333679, 'CAD': 1.237816, 'SGD': 1.346851}
2026-11-12 17:49:59.797861
__main__
INFO: Converting from EUR to USD: 52.217806271938734
2026-11-12 17:49:59.797917
__main__
INFO: Converting from USD to USD: 52.217806271938734
52.217806271938734


**Same converter, rewritten with `logging`.** Setup is done once at the top:

1. `logging.getLogger(__name__)` — get a **logger** named after the module.
2. `logging.StreamHandler(sys.stdout)` — a **handler** decides *where* records go (here, the console).
3. `logging.Formatter('[%(asctime)s] %(levelname)s [%(message)s]')` — a **formatter** decides *what a line looks like*; `%(asctime)s`, `%(levelname)s`, `%(message)s` are filled in automatically.
4. Attach the formatter to the handler and the handler to the logger, then `setLevel(logging.DEBUG)` so every level is shown.

After that, each three-print block collapses to one call: `logger.debug(...)`, `logger.info(...)`, `logger.error(...)`. The method picks the level, so the message no longer needs a hand-typed prefix. (The FROM-country error still carries a leftover `"ERROR:"` in its text, which would print as `ERROR [ERROR: ...]`.)

*Notebook gotcha:* re-running this cell calls `addHandler` again on the same logger, so every message prints twice (then three times...). Restart the kernel, or clear old handlers first, if that happens.

In [2]:
import logging
import sys

logger = logging.getLogger(__name__)
stream_handler = logging.StreamHandler(sys.stdout)
formatter = logging.Formatter('[%(asctime)s] %(levelname)s [%(message)s]')
stream_handler.setFormatter(formatter)
logger.addHandler(stream_handler)
logger.setLevel(logging.DEBUG)

exchange_rates = {
  "USD": 1.0,
  "EUR": 0.861775,
  "GBP": 0.726763,
  "INR": 75.054725,
  "AUD": 1.333679,
  "CAD": 1.237816,
  "SGD": 1.346851
}

def convert(from_country, to_country, amount):
  if to_country not in exchange_rates:
    logger.error("The TO country supplied is not a valid country.")
    sys.exit(0)
    
  if from_country not in exchange_rates:
    logger.error("ERROR: The FROM country supplied is not a valid country.")
    sys.exit(0)
        
  to_rate = exchange_rates[to_country]
  from_rate = exchange_rates[from_country]
    
  if from_country != "USD":
    converted_to_usd = amount / from_rate
    logger.info("Converting from {from_country} to USD: {converted_to_usd}".format(from_country=from_country, converted_to_usd=converted_to_usd))
        
    converted_from_usd = converted_to_usd * to_rate
    logger.info("Converting from USD to {to_country}: {converted_from_usd}".format(to_country=to_country, converted_from_usd=converted_from_usd))
        
    return converted_from_usd
        
  else:
    converted_from_usd = amount * to_rate
    logger.info("Converting from USD to {to_country}: {converted_from_usd}".format(to_country=to_country, converted_from_usd=converted_from_usd))
    return converted_from_usd
        
logger.debug("Current rates: {exchange_rates}".format(exchange_rates=exchange_rates))        
currency = convert("EUR", "USD", 45)
print(currency)
# -> [<asctime>] DEBUG [Current rates: {...}]
# -> [<asctime>] INFO [Converting from EUR to USD: 52.217806271938734]
# -> [<asctime>] INFO [Converting from USD to USD: 52.217806271938734]
# -> 52.217806271938734

[2026-11-12 17:49:59,812] DEBUG [Current rates: {'USD': 1.0, 'EUR': 0.861775, 'GBP': 0.726763, 'INR': 75.054725, 'AUD': 1.333679, 'CAD': 1.237816, 'SGD': 1.346851}]


[2026-11-12 17:49:59,813] INFO [Converting from EUR to USD: 52.217806271938734]


[2026-11-12 17:49:59,814] INFO [Converting from USD to USD: 52.217806271938734]


52.217806271938734


### Creating a Logger

Three steps: get a logger, make a handler, attach the handler.

**1. Get the logger.** `logging.getLogger(name)` returns a logger object.

- The same `name` always returns the **same** logger object.
- No name returns the **root logger**.
- Pass `__name__` (the current module's name) so each module gets its own logger and you don't accidentally grab someone else's.

```python
import logging
logger = logging.getLogger(__name__)
```

**2. Make a handler.** A **handler** decides *where* messages go. `logging.StreamHandler(stream)` writes to a stream. With no argument it goes to `sys.stderr`; pass `sys.stdout` to send it to standard console output (needs `import sys`).

```python
import sys
stream_handler = logging.StreamHandler(sys.stdout)
```

**3. Attach it.** `logger.addHandler(hdlr)` adds the handler to the logger.

```python
logger.addHandler(stream_handler)
```

In [3]:
# Same name -> same object; no name -> root logger
print(logging.getLogger(__name__) is logger)   # -> True
print(logging.getLogger().name)                # -> root
print(logger.name)                             # -> __main__

True
root
__main__


**Checkpoint (script.py):** import `logging` and `sys`; create `logger` named after the current module; create `stream_handler` pointed at the console and attach it. Nothing is logged yet, so the only output is the `print` of the logger's handler list.

*In this notebook the `__main__` logger already got a handler from the converter example above (same name → same logger), so the list shows two handlers here. In a fresh `script.py` it's just one.*

In [4]:
import logging
import sys

logger = logging.getLogger(__name__) 

stream_handler = logging.StreamHandler(sys.stdout)
logger.addHandler(stream_handler)
print(logger.handlers)
# -> [<StreamHandler stdout (NOTSET)>]   (fresh script; two entries in this notebook)

[<StreamHandler stdout (NOTSET)>, <StreamHandler stdout (NOTSET)>]


### Log Levels

Every message has a **level** = how severe it is. Each level is a constant in `logging` with a number; higher number = more severe.

| Level | Value | Use it for |
| --- | --- | --- |
| `NOTSET` | 0 | no level of its own — inherit from the nearest ancestor logger that has one |
| `DEBUG` | 10 | detailed info for debugging |
| `INFO` | 20 | normal, expected operations ("converting EUR to USD") |
| `WARNING` | 30 | something unexpected, current or coming, but the program keeps running |
| `ERROR` | 40 | a serious problem; some functionality is broken |
| `CRITICAL` | 50 | the worst; the program may stop entirely |

The numbers matter because a logger's level acts as a **threshold**: messages below it are dropped. That's why the converter example called `logger.setLevel(logging.DEBUG)`, so everything from 10 up gets through.

`NOTSET` per the lesson: the logger walks up to the first ancestor with a real level and uses that; with none, the threshold is 0. In practice the **root logger defaults to `WARNING`**, so a fresh logger with no level set only shows `WARNING` and above (checked below).

In [5]:
print(logging.DEBUG, logging.INFO, logging.WARNING, logging.ERROR, logging.CRITICAL)  # -> 10 20 30 40 50
print(logging.getLevelName(30))                                    # -> WARNING

fresh = logging.getLogger("fresh_demo")         # new logger, level never set
print(fresh.level)                              # -> 0   (NOTSET)
print(fresh.getEffectiveLevel())                # -> 30  (inherited from root = WARNING)

10 20 30 40 50
WARNING
0
30


**Checkpoint:** print the numeric value of each logging level, one `print()` per level.

In [6]:
import logging

print(logging.DEBUG)
print(logging.INFO)
print(logging.WARNING)
print(logging.ERROR)
print(logging.CRITICAL)
print(logging.NOTSET)
# -> 10
# -> 20
# -> 30
# -> 40
# -> 50
# -> 0

10
20
30
40
50
0


### Logging Errors and Messages

Each level has its own logger method; the method name *is* the level:

| Method | Level |
| --- | --- |
| `logger.debug(msg)` | DEBUG |
| `logger.info(msg)` | INFO |
| `logger.warning(msg)` | WARNING |
| `logger.error(msg)` | ERROR |
| `logger.critical(msg)` | CRITICAL |

`msg` is a string: a custom message, or the text of a caught error/exception.

There's also a generic **`logger.log(level, msg)`** where you pass the numeric level yourself:

```python
logger.log(logging.INFO, "This is an info message!")   # same as logger.info("This is an info message!")
```

(The lesson says this "works the same as the preceding `.debug()` example", but it's really equivalent to `.info()`, since the level passed is INFO.)

Below, a separate demo logger is used so the output isn't doubled by the two handlers already on `__main__`.

In [7]:
demo = logging.getLogger("methods_demo")
demo.addHandler(logging.StreamHandler(sys.stdout))
demo.setLevel(logging.DEBUG)

demo.debug("This is a debug message!")
demo.info("This is an info message!")
demo.warning("This is a warning message!")
demo.error("This is an error message!")
demo.critical("This is a critical message!")
demo.log(logging.INFO, "This is an info message via .log()!")
# -> each message printed once, in order (no formatter, so just the text)

This is a debug message!


This is an info message!


This is a warning message!


This is an error message!


This is a critical message!


This is an info message via .log()!


**Checkpoint (script.py), steps 1–4:** starter code sets up a console logger and a `division()` function that reads a dividend and divisor with `input()`. Each step adds one log call:

| Step | Where | Call | Test input |
| --- | --- | --- | --- |
| 1 | `divisor == 0` branch | `logger.error("Attempting to divide by 0!")` | 6, 0 |
| 2 | `except ValueError` | CRITICAL `"No dividend or divisor value entered!"` (written here as `logger.log(logging.CRITICAL, ...)`) | 6, blank |
| 3 | end of script | `if result == None:` → `logger.warning("The result value is None!")` | 6, 0 |
| 4 | after each `input()` | `logger.info(dividend)` / `logger.info(divisor)` (plus a `logger.debug` at the top) | 6, 3 |

Step 4's point: with 6 and 3 **nothing is logged**. No `setLevel` was called, so the threshold is the inherited `WARNING` (30); DEBUG (10) and INFO (20) are dropped. Only WARNING / ERROR / CRITICAL get through, which is why steps 1–3 showed up. The next sublesson fixes this.

`return` with no value returns `None`, so both failure paths make `result` None and trigger the step-3 warning.

*Notebook notes:* nbconvert can't type at an `input()` prompt, so the stand-in cell feeds the answers. It also resets the `__main__` logger, clearing the handlers earlier cells stacked on it and undoing the converter cell's `setLevel(DEBUG)` (same name → same logger object, so that level was still in effect), so output matches a fresh `script.py`.

In [8]:
# Notebook stand-in: start the __main__ logger fresh (like a new script.py),
# and answer the input() prompts for the step-4 run: 6, then 3
logging.getLogger(__name__).handlers.clear()
logging.getLogger(__name__).setLevel(logging.NOTSET)   # undo the converter cell's setLevel(DEBUG)

answers = iter(["6", "3"])

def input(prompt):
    value = next(answers)
    print(prompt + value)
    return value

In [9]:
import logging
import sys

logger = logging.getLogger(__name__)
stream_handler = logging.StreamHandler(sys.stdout)
logger.addHandler(stream_handler)
        
def division():
  logger.debug("Starting Division! ")
  try:
    dividend = float(input("Enter the dividend: "))
    logger.info(dividend)
    divisor = float(input("Enter the divisor: "))
    logger.info(divisor)
  except ValueError:
    logger.log(logging.CRITICAL, "No dividend or divisor value entered!")
    return
  if divisor == 0:
    logger.error("Attempting to divide by 0!")
    return
  else:
    return dividend/divisor
result = division()
if result == None:
  logger.warning("The result value is None!")
# -> Enter the dividend: 6
# -> Enter the divisor: 3
# (no log lines: DEBUG and INFO are below the default WARNING threshold; result = 2.0)

Enter the dividend: 6
Enter the divisor: 3


In [10]:
# Steps 1-3 scenarios again with the same function
answers = iter(["6", "0"])          # divide by zero
result = division()
if result == None:
  logger.warning("The result value is None!")
# -> Attempting to divide by 0!
# -> The result value is None!

answers = iter(["6", ""])           # blank divisor
result = division()
if result == None:
  logger.warning("The result value is None!")
# -> No dividend or divisor value entered!
# -> The result value is None!

Enter the dividend: 6
Enter the divisor: 0
Attempting to divide by 0!


The result value is None!


Enter the dividend: 6
Enter the divisor: 
No dividend or divisor value entered!


The result value is None!


### Setting the Log Level

Why didn't the DEBUG/INFO messages show in the last checkpoint? Every logger has a **threshold**: only messages *at or above* it get output. The default is **WARNING**, so DEBUG (10) and INFO (20) were filtered out.

Change it with `logger.setLevel(level)`, where `level` is one of the numeric constants:

```python
logger.setLevel(logging.DEBUG)   # now DEBUG and everything above shows
```

When to pick which:

- **DEBUG** during development, when you want every helpful debug message.
- **WARNING** once deployed, to hide the developer-only DEBUG/INFO chatter.
- **CRITICAL** when tracking down a crash, to see only the most severe messages.


In [11]:
lvl = logging.getLogger("level_demo")
lvl.addHandler(logging.StreamHandler(sys.stdout))

for threshold in [logging.WARNING, logging.DEBUG, logging.CRITICAL]:
  lvl.setLevel(threshold)
  print("--- level set to", logging.getLevelName(threshold))
  lvl.debug("debug msg")
  lvl.info("info msg")
  lvl.warning("warning msg")
  lvl.critical("critical msg")
# -> WARNING:  warning msg, critical msg
# -> DEBUG:    all four
# -> CRITICAL: critical msg only


--- level set to WARNING
warning msg


critical msg


--- level set to DEBUG
debug msg


info msg


warning msg


critical msg


--- level set to CRITICAL
critical msg


**Checkpoint (script.py):** same calculator, with one new line after the handler setup: `logger.setLevel(logging.DEBUG)`. Run it with 6 and 3. This time **every** log line shows, including the DEBUG and INFO ones that were silent before.

Note: this version catches `except SyntaxError` instead of the earlier `except ValueError`. A blank input makes `float("")` raise a **ValueError**, so this `except` won't catch it and the 6-then-blank case would crash with a traceback. The 6-then-3 run here doesn't hit that path.


In [12]:
# Notebook stand-in: fresh __main__ logger again (clear the earlier cells' handler), answer 6 then 3
logging.getLogger(__name__).handlers.clear()
answers = iter(["6", "3"])


In [13]:
import logging
import sys

logger = logging.getLogger(__name__)
stream_handler = logging.StreamHandler(sys.stdout)
logger.addHandler(stream_handler)
logger.setLevel(logging.DEBUG)

def division():
  logger.debug("Starting Division!")
  try:
    dividend = float(input("Enter the dividend: "))
    logger.info(dividend)
    divisor = float(input("Enter the divisor: "))
    logger.info(divisor)
  except SyntaxError:
    logger.log(logging.CRITICAL, "No dividend or divisor value entered!")
    return
  if divisor == 0:
    logger.error("Attempting to divide by 0!")
    return
  else:
      return dividend/divisor
result = division()
if result == None:
  logger.warning("The result value is None!")
# -> Starting Division!
# -> Enter the dividend: 6
# -> 6.0
# -> Enter the divisor: 3
# -> 3.0
# (result = 2.0, so no warning)


Starting Division!


Enter the dividend: 6
6.0


Enter the divisor: 3
3.0


### Logging to a File

Console logs vanish once the program ends. Sending logs to a **file** keeps them afterwards, so you can search them later or feed them into log tools like Splunk or an ELK Stack.

Use `logging.FileHandler(filename)`. It's another handler, just like `StreamHandler`, but its destination is a file. Attach it the same way:

```python
file_handler = logging.FileHandler("my_program.log")
logger.addHandler(file_handler)
```

A logger can have both handlers at once: one message, two destinations (console + file). By default `FileHandler` **appends**, so re-running the program adds to the existing file rather than replacing it.


In [14]:
import os

fl = logging.getLogger("file_demo")
fl.setLevel(logging.DEBUG)
file_handler = logging.FileHandler("file_demo.log")
fl.addHandler(file_handler)

fl.info("saved to the file")
fl.warning("this too")
file_handler.close()                 # flush and release the file

print(open("file_demo.log").read())
os.remove("file_demo.log")           # tidy up so the notes folder stays clean
# -> saved to the file
# -> this too


saved to the file
this too



**Checkpoint (script.py):** create `file_handler = logging.FileHandler("output.log")` and attach it with `addHandler`. The `StreamHandler` is gone from this version, so with inputs 6 and 3 the **console shows only the `input()` prompts** and every log line goes into `output.log` instead.

*Notebook notes:* the stand-in clears the `__main__` logger's earlier handler and feeds 6, 3. A final cell prints `output.log`, then closes the handler and deletes the file so the notes folder stays clean.


In [15]:
# Notebook stand-in: fresh __main__ logger, answer 6 then 3
logging.getLogger(__name__).handlers.clear()
answers = iter(["6", "3"])


In [16]:
import logging
import sys

logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
file_handler = logging.FileHandler("output.log")
logger.addHandler(file_handler)
        
def division():
  logger.debug("Starting Division!")
  try:
    dividend = float(input("Enter the dividend: "))
    logger.info(dividend)
    divisor = float(input("Enter the divisor: "))
    logger.info(divisor)
  except SyntaxError:
    logger.log(logging.CRITICAL, "No dividend or divisor value entered!")
    return
  if divisor == 0:
    logger.error("Attempting to divide by 0!")
    return
  else:
      return dividend/divisor

result = division()
if result == None:
  logger.warning("The result value is None!")
# -> Enter the dividend: 6
# -> Enter the divisor: 3
# (no log lines in the console: they all went to output.log)


Enter the dividend: 6
Enter the divisor: 3


In [17]:
# What's in output.log (the "open it in the editor" step)
print(open("output.log").read())
# -> Starting Division!
# -> 6.0
# -> 3.0

# Notebook cleanup: detach + close the handler, delete the file
logger.removeHandler(file_handler)
file_handler.close()
os.remove("output.log")


Starting Division!
6.0
3.0



### Logging to the Console and a File

One logger can hold **several handlers**, so each message can go to several places at once, such as the console and a file. Just call `addHandler` once per handler:

```python
logger.addHandler(file_handler)
logger.addHandler(stream_handler)
```

Typical use: show important things like errors on the console, and write everything (including debug detail) to a file you can search afterwards.

*Beyond the lesson text:* that split works because **handlers have their own `setLevel`** too. A message must pass the logger's level first, then each handler's level. So set the logger to DEBUG, the file handler to DEBUG, and the console handler to ERROR.


In [18]:
both = logging.getLogger("both_demo")
both.setLevel(logging.DEBUG)                  # logger lets everything through

console = logging.StreamHandler(sys.stdout)
console.setLevel(logging.ERROR)               # console: errors and above only
to_file = logging.FileHandler("both_demo.log")
to_file.setLevel(logging.DEBUG)               # file: everything

both.addHandler(to_file)
both.addHandler(console)

both.debug("debug detail")
both.info("info detail")
both.error("something broke")
# console -> something broke

to_file.close()
print("--- both_demo.log:")
print(open("both_demo.log").read())
os.remove("both_demo.log")
# -> debug detail
# -> info detail
# -> something broke


something broke


--- both_demo.log:
debug detail
info detail
something broke



**Checkpoint (script.py):** attach two handlers to the same logger: `file_handler` → `calculator.log` and `stream_handler` → `sys.stdout`. Run with 6 and 0. The divide-by-zero path logs on both, so the **console output and `calculator.log` contain the same log lines**. Neither handler sets its own level, so both pass everything the logger's DEBUG threshold allows.

*Notebook notes:* same stand-in as before (clear the `__main__` handlers, feed 6, 0). A final cell prints `calculator.log`, then closes the file handler and deletes the file.


In [19]:
# Notebook stand-in: fresh __main__ logger, answer 6 then 0
logging.getLogger(__name__).handlers.clear()
answers = iter(["6", "0"])


In [20]:
import logging
import sys

logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
file_handler = logging.FileHandler("calculator.log")
logger.addHandler(file_handler)
stream_handler = logging.StreamHandler(sys.stdout) 
logger.addHandler(stream_handler)

        

def division():
  logger.debug("Starting Division!")
  try:
    dividend = float(input("Enter the dividend: "))
    logger.info(dividend)
    divisor = float(input("Enter the divisor: "))
    logger.info(divisor)
  except SyntaxError:
    logger.log(logging.CRITICAL, "No dividend or divisor value entered!")
    return
  if divisor == 0:
    logger.error("Attempting to divide by 0!")
    return
  else:
    return dividend/divisor

result = division()
if result == None:
  logger.warning("The result value is None!")
# -> Starting Division!
# -> Enter the dividend: 6
# -> 6.0
# -> Enter the divisor: 0
# -> 0.0
# -> Attempting to divide by 0!
# -> The result value is None!


Starting Division!


Enter the dividend: 6
6.0


Enter the divisor: 0
0.0


Attempting to divide by 0!


The result value is None!


In [21]:
# What's in calculator.log: same log lines as the console (minus the input() prompts)
print(open("calculator.log").read())
# -> Starting Division!
# -> 6.0
# -> 0.0
# -> Attempting to divide by 0!
# -> The result value is None!

# Notebook cleanup: detach + close the file handler, delete the file
logger.removeHandler(file_handler)
file_handler.close()
os.remove("calculator.log")


Starting Division!
6.0
0.0
Attempting to divide by 0!
The result value is None!



### Formatting the Logs

With no formatter, a handler outputs only the bare message (everything so far). A **`Formatter`** adds context to every line. Its first argument is a format string built from placeholders:

| Placeholder | Gives |
| --- | --- |
| `%(asctime)s` | date and time, e.g. `2026-01-29 14:16:41,211` (the `,211` is milliseconds) |
| `%(levelname)s` | level name, e.g. `WARNING` |
| `%(name)s` | the **logger's** name (`__main__` when run directly) |
| `%(lineno)d` | line number of the logging call (an integer, hence `d`) |
| `%(message)s` | the message itself |

The trailing `s` / `d` mean string / integer, the old `%`-formatting style.

Attach it to a **handler** with `handler.setFormatter(formatter)`:

```python
formatter = logging.Formatter("[%(asctime)s] %(levelname)s:%(name)s:%(lineno)d:%(message)s")
stream_handler.setFormatter(formatter)
logger.warning("This is a warning!")
# -> [2026-01-29 14:21:44,359] WARNING:__main__:9:This is a warning!
```

The formatter belongs to the handler, not the logger. So with two handlers you can give each its own format, for example a detailed format on the console for errors and a lean one in the file. More placeholders are listed in the Python `logging` docs (LogRecord attributes).


In [22]:
fmt = logging.getLogger("fmt_demo")
fmt.setLevel(logging.DEBUG)

detailed = logging.StreamHandler(sys.stdout)
detailed.setFormatter(logging.Formatter("[%(asctime)s] %(levelname)s:%(name)s:%(lineno)d:%(message)s"))
lean = logging.StreamHandler(sys.stdout)
lean.setFormatter(logging.Formatter("%(levelname)s - %(message)s"))

fmt.addHandler(detailed)
fmt.addHandler(lean)

fmt.warning("This is a warning!")   # one message, formatted twice (once per handler)
# -> [<today's date + time>] WARNING:fmt_demo:12:This is a warning!   (12 = line of the call in this cell)
# -> WARNING - This is a warning!


[2026-11-12 17:50:00,007] WARNING:fmt_demo:12:This is a warning!


WARNING - This is a warning!


**Checkpoint (script.py):** two handlers, two formats. The checkpoint's strings are **not** the reading's example, and the checker compares them character by character:

| Handler | Formatter | Format string |
| --- | --- | --- |
| `file_handler` → `formatted.log` | `formatter1` | `[%(asctime)s] {%(levelname)s} %(name)s: #%(lineno)d - %(message)s` |
| `stream_handler` → console | `formatter2` | `[%(asctime)s] {%(levelname)s} - %(message)s` |

Then run it a few times with different inputs. The file lines carry the logger name and line number; the console lines don't.

*Notebook notes:* the stand-in clears the `__main__` handlers and feeds 6, 3. A second cell reruns `division()` with 6, 0 (the "a few times" step). The last cell prints `formatted.log`, then closes the file handler and deletes the file. Timestamps and line numbers will differ in a real `script.py`.


In [23]:
# Notebook stand-in: fresh __main__ logger, answer 6 then 3
logging.getLogger(__name__).handlers.clear()
answers = iter(["6", "3"])


In [24]:
import logging
import sys

logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
file_handler = logging.FileHandler("formatted.log")
stream_handler = logging.StreamHandler(sys.stdout)
logger.addHandler(file_handler)
logger.addHandler(stream_handler)

formatter1 = logging.Formatter("[%(asctime)s] {%(levelname)s} %(name)s: #%(lineno)d - %(message)s")
formatter2 = logging.Formatter("[%(asctime)s] {%(levelname)s} - %(message)s")
file_handler.setFormatter(formatter1)
stream_handler.setFormatter(formatter2)


def division():
  logger.debug("Starting Division!")
  try:
    dividend = float(input("Enter the dividend: "))
    logger.info(dividend)
    divisor = float(input("Enter the divisor: "))
    logger.info(divisor)
  except SyntaxError:
    logger.log(logging.CRITICAL, "No dividend or divisor value entered!")
    return
  if divisor == 0:
    logger.error("Attempting to divide by 0!")
    return
  else:
    return dividend/divisor

result = division()
if result == None:
  logger.warning("The result value is None!")
# -> [<time>] {DEBUG} - Starting Division!
# -> Enter the dividend: 6
# -> [<time>] {INFO} - 6.0
# -> Enter the divisor: 3
# -> [<time>] {INFO} - 3.0


[2026-11-12 17:50:00,031] {DEBUG} - Starting Division!


Enter the dividend: 6


[2026-11-12 17:50:00,032] {INFO} - 6.0


Enter the divisor: 3
[2026-11-12 17:50:00,033] {INFO} - 3.0


In [25]:
# Second run: 6 then 0
answers = iter(["6", "0"])
result = division()
if result == None:
  logger.warning("The result value is None!")
# -> ... same three lines with 0.0, then
# -> [<time>] {ERROR} - Attempting to divide by 0!
# -> [<time>] {WARNING} - The result value is None!


[2026-11-12 17:50:00,045] {DEBUG} - Starting Division!


Enter the dividend: 6
[2026-11-12 17:50:00,046] {INFO} - 6.0


Enter the divisor: 0
[2026-11-12 17:50:00,048] {INFO} - 0.0


[2026-11-12 17:50:00,049] {ERROR} - Attempting to divide by 0!


[2026-11-12 17:50:00,050] {WARNING} - The result value is None!


In [26]:
# formatted.log: same messages as the console, but with name + line number
print(open("formatted.log").read())
# -> [<time>] {DEBUG} __main__: #18 - Starting Division!
# -> [<time>] {INFO} __main__: #21 - 6.0
# -> ...
# -> [<time>] {ERROR} __main__: #28 - Attempting to divide by 0!
# -> [<time>] {WARNING} __main__: #5 - The result value is None!
#    (#5 because that warning call sits on line 5 of the rerun cell: lineno = where the call is)

# Notebook cleanup: detach + close the file handler, delete the file
logger.removeHandler(file_handler)
file_handler.close()
os.remove("formatted.log")


[2026-11-12 17:50:00,031] {DEBUG} __main__: #18 - Starting Division!
[2026-11-12 17:50:00,032] {INFO} __main__: #21 - 6.0
[2026-11-12 17:50:00,033] {INFO} __main__: #23 - 3.0
[2026-11-12 17:50:00,045] {DEBUG} __main__: #18 - Starting Division!
[2026-11-12 17:50:00,046] {INFO} __main__: #21 - 6.0
[2026-11-12 17:50:00,048] {INFO} __main__: #23 - 0.0
[2026-11-12 17:50:00,049] {ERROR} __main__: #28 - Attempting to divide by 0!
[2026-11-12 17:50:00,050] {WARNING} __main__: #5 - The result value is None!



### Using basicConfig()

`logging.basicConfig()` does the whole setup (level, handler, format) in **one call**:

```python
logging.basicConfig(filename='calculator.log', level=logging.DEBUG,
                    format='[%(asctime)s] %(levelname)s - %(message)s')
```

That one line adds a `FileHandler` for `calculator.log`, sets the level to DEBUG, and applies the format. You still get your logger the usual way, with `logging.getLogger(__name__)`.

With **no arguments**, `basicConfig()` adds a `StreamHandler` to `sys.stderr` with the default format `%(levelname)s:%(name)s:%(message)s`.

*Beyond the lesson text*, two gotchas worth knowing:

- `basicConfig` configures the **root logger**. Your `getLogger(__name__)` logger has no handlers of its own; its messages **propagate** up to the root logger, which outputs them.
- It only works **once**. If the root logger already has handlers, later `basicConfig` calls silently do nothing. Pass `force=True` to replace the old setup (handy in Jupyter, where re-running cells is common).


In [27]:
# 1) The lesson's one-liner (force=True because this is a notebook)
logging.basicConfig(filename='basic_demo.log', level=logging.DEBUG,
                    format='[%(asctime)s] %(levelname)s - %(message)s', force=True)
bc = logging.getLogger("basic_demo")       # no handlers of its own...
bc.debug("debug via basicConfig")
bc.error("error via basicConfig")
print(bc.handlers, logging.getLogger().handlers)   # ...the FileHandler is on the root logger
print(open("basic_demo.log").read())
# -> [] [<FileHandler .../basic_demo.log (NOTSET)>]
# -> [<time>] DEBUG - debug via basicConfig
# -> [<time>] ERROR - error via basicConfig

# 2) The default format (stream=sys.stdout only so the notebook shows it; the real default is stderr)
logging.basicConfig(stream=sys.stdout, force=True)
bc.warning("default format")
# -> WARNING:basic_demo:default format

# Notebook cleanup: remove the root handler, restore the root's default WARNING level, delete the file
logging.getLogger().handlers.clear()   # (part 2's force=True already closed the file handler)
logging.getLogger().setLevel(logging.WARNING)
os.remove("basic_demo.log")


[] [<FileHandler /home/plewis/GitHub_Repos/Self_Study_and_Projects/Codecademy/ML_AI_Career_Path/Notes/SWE-ML/basic_demo.log (NOTSET)>]
[2026-11-12 17:50:00,075] DEBUG - debug via basicConfig
[2026-11-12 17:50:00,075] ERROR - error via basicConfig



**Checkpoint (script.py):** replace the manual handler setup with one `basicConfig` call: file `basic_config.log`, level **INFO**, format `[%(asctime)s] %(levelname)s - %(message)s`. Run it with 0 and 0. The file has the INFO, ERROR and WARNING lines but **no DEBUG line**: `"Starting Division!"` is DEBUG (10), below the INFO (20) threshold.

Why INFO applies even though the level was set on the root: `logger` itself is NOTSET, so its effective level comes from its first configured ancestor, the root logger, which `basicConfig` set to INFO.

*Notebook notes:* the stand-in resets the `__main__` logger to a fresh state (clears its handlers **and** sets it back to NOTSET; the earlier `setLevel(DEBUG)` would otherwise override the root's INFO and let DEBUG through) and feeds 0, 0. The last cell prints `basic_config.log`, then clears the root handler and deletes the file.


In [28]:
# Notebook stand-in: fresh __main__ logger (no handlers, NOTSET), answer 0 then 0
logging.getLogger(__name__).handlers.clear()
logging.getLogger(__name__).setLevel(logging.NOTSET)
answers = iter(["0", "0"])


In [29]:
import logging
import sys

logger = logging.getLogger(__name__)
logging.basicConfig(filename = "basic_config.log", level = logging.INFO, format = "[%(asctime)s] %(levelname)s - %(message)s")

        
def division():
  logger.debug("Starting Division!")
  try:
    dividend = float(input("Enter the dividend: "))
    logger.info(dividend)
    divisor = float(input("Enter the divisor: "))
    logger.info(divisor)
  except SyntaxError:
    logger.log(logging.CRITICAL, "No dividend or divisor value entered!")
    return
  if divisor == 0:
    logger.error("Attempting to divide by 0!")
    return
  else:
    return dividend/divisor

result = division()
if result == None:
  logger.warning("The result value is None!")
# -> Enter the dividend: 0
# -> Enter the divisor: 0
# (no log lines in the console: basicConfig's only handler is the file)


Enter the dividend: 0


Enter the divisor: 0


In [30]:
# basic_config.log: lesson format, and no DEBUG line
print(open("basic_config.log").read())
# -> [<time>] INFO - 0.0
# -> [<time>] INFO - 0.0
# -> [<time>] ERROR - Attempting to divide by 0!
# -> [<time>] WARNING - The result value is None!

# Notebook cleanup: close + remove the root's file handler, restore root to WARNING, delete the file
root = logging.getLogger()
for h in root.handlers:
  h.close()
root.handlers.clear()
root.setLevel(logging.WARNING)
os.remove("basic_config.log")


[2026-11-12 17:50:00,097] INFO - 0.0
[2026-11-12 17:50:00,098] INFO - 0.0
[2026-11-12 17:50:00,098] ERROR - Attempting to divide by 0!
[2026-11-12 17:50:00,098] WARNING - The result value is None!



## Q & A (captured as I go)

*Questions posed during the lesson + answers, folded in per sublesson.*

**Q: What are a handler, `StreamHandler`, and `addHandler`, in simple terms?**

Think of the logger as someone writing notes and the handler as the **delivery route** for each note.

- **Logger** — the thing you call (`logger.info("...")`). It creates the message but doesn't decide where it ends up.
- **Handler** — decides *where* the message goes: the screen, a file, an email, etc. No handler, nowhere useful to go.
- **`StreamHandler`** — one specific kind of handler that writes to a **stream**, i.e. a text output channel. `StreamHandler(sys.stdout)` prints to regular console output; with no argument it uses `sys.stderr` (the error channel, often shown in red in notebooks). Other kinds exist, e.g. `FileHandler("app.log")` writes to a file.
- **`addHandler`** — plugs the handler into the logger: `logger.addHandler(stream_handler)`. A logger can have several handlers, so one message can go to the console *and* a file at once.

In one line: the logger makes the message, the handler delivers it, and `addHandler` connects the two.

**Follow-up: so `addHandler` just sends the message back to me?**

Not quite. `addHandler` doesn't send anything; it's a **one-time setup** step that hooks the handler onto the logger. The sending happens later, every time you call `logger.info(...)` / `.debug(...)` / `.error(...)`: the logger passes the message to each attached handler, and the handler writes it to its destination. With `StreamHandler(sys.stdout)` that destination is your console, which is why it *looks* like the message comes back to you. With a `FileHandler` it would land in a file instead and you'd see nothing on screen.

| step | when it runs | what it does |
| --- | --- | --- |
| `logger.addHandler(h)` | once, at setup | attaches handler `h` |
| `logger.info("...")` | every log call | creates the message, hands it to `h` |
| `h` (the handler) | automatically | writes it to stdout / a file / etc. |

**Mental model:** `addHandler` builds a **bridge** between the logger and the handler. Building it happens once; messages cross it every time you log. Each `addHandler` call builds another bridge, so a logger can feed several destinations (and re-running the setup cell builds a duplicate bridge, which is why output doubles).

**Q: Is `__name__` a dunder method that creates the name value / object?**

It's a dunder, but a **variable**, not a method — no parentheses, nothing gets called or created. Python sets it automatically on every module, holding that module's name as a string:

- file run directly (`python converter.py`, or a notebook) → `__name__ == "__main__"`
- file imported (`import converter`) → `__name__ == "converter"`

So `logging.getLogger(__name__)` just passes a string like `"__main__"` or `"converter"`. `getLogger` is what creates (or looks up) the logger object under that name. Using `__name__` means each module automatically gets a logger labeled with its own name.

Same idea behind the classic `if __name__ == "__main__":` guard — "only run this when the file is executed directly, not when imported." Contrast with dunder *methods* like `__init__` or `__next__`, which are functions Python calls for you.


**Q: What is `try` / `except`? Is it another conditional?**

Not quite. It's **exception handling**, not a conditional.

- `if` checks a condition *before* running code: "is this true? then do X."
- `try` just *runs* the code. If something in it raises an error (an **exception**), Python jumps straight to the matching `except` block instead of crashing the program.

```python
try:
    x = int(input("Enter a number: "))   # blank or "abc" -> raises ValueError
    print(10 / x)                        # 0 -> raises ZeroDivisionError
except ValueError:
    print("That wasn't a number.")
except ZeroDivisionError:
    print("Can't divide by 0.")
```

If nothing goes wrong, the `except` blocks are skipped. Use `try`/`except` when you can't easily check ahead of time (bad user input, a missing file, a network failure). That's why `division()` logs CRITICAL inside `except ValueError`: a blank input makes `int("")` fail, and the `except` catches it so you can log it.


**Q: What do dunders do, in general?**

Dunder (double-underscore) names are **Python's own reserved hooks**. The double underscores mark them as "Python uses this behind the scenes," so they don't collide with your names. Two kinds:

1. **Dunder attributes**: info Python fills in for you. `__name__` (module name), `__file__` (file path), `__doc__` (docstring).
2. **Dunder methods**: you define them in a class, and Python calls them for you when you use normal syntax.

| You write | Python actually calls |
| --- | --- |
| `Point(1, 2)` | `__init__` |
| `print(p)` / `str(p)` | `__str__` |
| `len(p)` | `__len__` |
| `p + q` | `__add__` |
| `p == q` | `__eq__` |

```python
class Point:
    def __init__(self, x, y):
        self.x, self.y = x, y
    def __str__(self):
        return f"({self.x}, {self.y})"
    def __add__(self, other):
        return Point(self.x + other.x, self.y + other.y)

print(Point(1, 2) + Point(3, 4))   # -> (4, 6)
```

You rarely call dunders directly (`p.__add__(q)`). You write `p + q` and Python routes it to the dunder.


**Q: Is a module "above" a class?**

Yes. A **module** is just a `.py` file, and it *contains* classes, functions, and variables. Top to bottom:

```
package     folder of modules           (logging/)
 └ module   one .py file                (logging/__init__.py, or your script.py)
    └ class / function / variable       (Logger, StreamHandler, getLogger, DEBUG)
       └ method / attribute             (logger.addHandler, logger.level)
```

So `logging.StreamHandler(sys.stdout)` reads as: from the **module** `logging`, take the **class** `StreamHandler` and make an object from it. And `logger.addHandler(...)` calls a **method** on that object. `__name__` lives at the module level, which is why it gives the module's name rather than a class's.


**Q: Can `FileHandler` write to a `.txt` file?**

Yes. `FileHandler("my_program.txt")` works exactly like `FileHandler("my_program.log")`. Either way the output is plain text, one line per message. The `.log` extension is only a convention: it signals "this is a log file" to people and to log tools like Splunk and ELK. The file is created if it doesn't exist yet, and appended to if it does.


**Q: Is there a C++ equivalent of `logging`?**

Not in the standard library. C++ only gives you raw streams: `std::cout` (stdout), `std::cerr` (stderr, unbuffered), and `std::clog` (stderr, buffered, meant for logs), with no levels or formatting. Real logging comes from libraries:

| Library | Notes |
| --- | --- |
| **spdlog** | Most popular. Header-only, fast, closest feel to Python's `logging` |
| glog | Google's logger |
| Boost.Log | Very configurable, heavy |
| **ROOT** `Info` / `Warning` / `Error` / `Fatal` | Built into ROOT. `gErrorIgnoreLevel = kWarning;` plays the role of `setLevel` |

spdlog mapped onto the Python pieces:

```cpp
#include "spdlog/spdlog.h"
#include "spdlog/sinks/basic_file_sink.h"

auto logger = spdlog::basic_logger_mt("calc", "my_program.log");  // getLogger + FileHandler
logger->set_level(spdlog::level::debug);                            // setLevel(DEBUG)
logger->info("dividend = {}", 6.0);                                 // logger.info(...)
logger->error("Attempting to divide by 0!");                        // logger.error(...)
```

Same ideas, different names: spdlog's **sinks** are Python's **handlers**. Its levels are trace, debug, info, warn, err, critical.


## TL;DR

- **Why `logging` over `print()`:** levels, timestamps and module names come built in; messages can go to the console, a file, or both; and you turn noise down with one setting instead of deleting prints before shipping.
- **Logger → handler → formatter.** `logger = logging.getLogger(__name__)` makes the messages (same name, same object). A **handler** delivers them: `StreamHandler(sys.stdout)` for the console, `FileHandler("app.log")` for a file. Attach with `logger.addHandler(h)`, once per handler.
- **Levels:** DEBUG 10 < INFO 20 < WARNING 30 < ERROR 40 < CRITICAL 50. Log with `logger.debug/info/warning/error/critical(msg)` or `logger.log(level, msg)`.
- **Threshold:** only messages at or above the level get out. The default is **WARNING**, which is why DEBUG and INFO were silent until `logger.setLevel(logging.DEBUG)`. Use DEBUG in development and WARNING in production.
- **Files** persist after the program ends (searchable, can feed Splunk/ELK). `FileHandler` appends by default. Any extension works; `.log` is the convention.
- **Multiple handlers:** one message, many destinations. Each handler can have its own `setLevel` (e.g. console = ERROR, file = DEBUG).
- **Formatting:** `handler.setFormatter(logging.Formatter("[%(asctime)s] %(levelname)s:%(name)s:%(lineno)d:%(message)s"))`. The formatter belongs to the handler, so each handler can look different. `lineno` is the line of the logging call.
- **`basicConfig(filename=..., level=..., format=...)`** does the setup in one line, on the **root** logger. Named loggers propagate up to it and inherit its level when NOTSET. It only takes effect once unless `force=True`.
- **Jupyter gotcha:** re-running setup cells stacks duplicate handlers (doubled output) and leftover levels stick to the same-named logger. Clear handlers or restart the kernel.
